In [1]:
import pandas as pd
import numpy as np

import re
import pickle
import os

from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

from scipy.sparse import hstack, csr_matrix, save_npz

In [2]:
df = pd.read_csv(
    "../data/raw/DOB_ECB_Violations_20260913 (1).csv"
)

print(df.shape)

(102467, 46)


C:\Users\gimha_9pk7du7\AppData\Local\Temp\ipykernel_23464\913590179.py:1: DtypeWarning: Columns (28: SECTION_LAW_DESCRIPTION3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


In [3]:
df.columns.tolist()

['ISN_DOB_BIS_EXTRACT',
 'ECB_VIOLATION_NUMBER',
 'ECB_VIOLATION_STATUS',
 'DOB_VIOLATION_NUMBER',
 'BIN',
 'BORO',
 'BLOCK',
 'LOT',
 'HEARING_DATE',
 'HEARING_TIME',
 'SERVED_DATE',
 'ISSUE_DATE',
 'SEVERITY',
 'VIOLATION_TYPE',
 'RESPONDENT_NAME',
 'RESPONDENT_HOUSE_NUMBER',
 'RESPONDENT_STREET',
 'RESPONDENT_CITY',
 'RESPONDENT_ZIP',
 'VIOLATION_DESCRIPTION',
 'PENALITY_IMPOSED',
 'AMOUNT_PAID',
 'BALANCE_DUE',
 'INFRACTION_CODE1',
 'SECTION_LAW_DESCRIPTION1',
 'INFRACTION_CODE2',
 'SECTION_LAW_DESCRIPTION2',
 'INFRACTION_CODE3',
 'SECTION_LAW_DESCRIPTION3',
 'INFRACTION_CODE4',
 'SECTION_LAW_DESCRIPTION4',
 'INFRACTION_CODE5',
 'SECTION_LAW_DESCRIPTION5',
 'INFRACTION_CODE6',
 'SECTION_LAW_DESCRIPTION6',
 'INFRACTION_CODE7',
 'SECTION_LAW_DESCRIPTION7',
 'INFRACTION_CODE8',
 'SECTION_LAW_DESCRIPTION8',
 'INFRACTION_CODE9',
 'SECTION_LAW_DESCRIPTION9',
 'INFRACTION_CODE10',
 'SECTION_LAW_DESCRIPTION10',
 'AGGRAVATED_LEVEL',
 'HEARING_STATUS',
 'CERTIFICATION_STATUS']

In [4]:
#convert ISSUE_DATE to datetime
df["ISSUE_DATE"] = pd.to_datetime(
    df["ISSUE_DATE"],
    format="%Y%m%d",
    errors="coerce"
)

In [5]:
df["ISSUE_DATE"].head()

0   2026-06-01
1   2025-08-05
2   2026-06-24
3   2025-12-03
4   2026-07-22
Name: ISSUE_DATE, dtype: datetime64[us]

In [6]:
df = df.sort_values(
    [
        "BIN",
        "ISSUE_DATE"
    ]
)

In [7]:
#Previous violation count
df["PREVIOUS_VIOLATION_COUNT"] = (
    df.groupby("BIN")
    .cumcount()
)

In [8]:
#pervious class count 1
df["PREVIOUS_CLASS1_COUNT"] = (
    df.groupby("BIN")["SEVERITY"]
    .apply(
        lambda x:
        x.shift()
        .eq("CLASS - 1")
        .cumsum()
    )
    .reset_index(level=0, drop=True)
)

In [10]:
#pervious class count 2
df["PREVIOUS_CLASS2_COUNT"] = (
    df.groupby("BIN")["SEVERITY"]
    .apply(
        lambda x:
        x.shift()
        .eq("CLASS - 2")
        .cumsum()
    )
    .reset_index(level=0, drop=True)
)

In [11]:
#pervious class count 3
df["PREVIOUS_CLASS3_COUNT"] = (
    df.groupby("BIN")["SEVERITY"]
    .apply(
        lambda x:
        x.shift()
        .eq("CLASS - 3")
        .cumsum()
    )
    .reset_index(level=0, drop=True)
)

In [12]:
#check
df[
[
"BIN",
"SEVERITY",
"PREVIOUS_VIOLATION_COUNT",
"PREVIOUS_CLASS1_COUNT",
"PREVIOUS_CLASS2_COUNT",
"PREVIOUS_CLASS3_COUNT"
]
].head(10)

,BIN,SEVERITY,PREVIOUS_VIOLATION_COUNT,PREVIOUS_CLASS1_COUNT,PREVIOUS_CLASS2_COUNT,PREVIOUS_CLASS3_COUNT
17523,1000000.0,CLASS - 2,0.0,0.0,0.0,0.0
40875,1000000.0,CLASS - 2,1.0,0.0,1.0,0.0
78310,1000000.0,CLASS - 2,2.0,0.0,2.0,0.0
9293,1000000.0,CLASS - 2,3.0,0.0,3.0,0.0
4111,1000005.0,CLASS - 2,0.0,0.0,0.0,0.0
7163,1000005.0,CLASS - 2,1.0,0.0,1.0,0.0
10840,1000007.0,CLASS - 1,0.0,0.0,0.0,0.0
11999,1000007.0,CLASS - 2,1.0,1.0,0.0,0.0
8191,1000007.0,CLASS - 2,2.0,1.0,1.0,0.0
25601,1000007.0,CLASS - 1,3.0,1.0,2.0,0.0


In [69]:
#Create Date Features
df["ISSUE_YEAR"] = (
    df["ISSUE_DATE"].dt.year
)

df["ISSUE_MONTH"] = (
    df["ISSUE_DATE"].dt.month
)

df["ISSUE_DAY"] = (
    df["ISSUE_DATE"].dt.day
)

df["ISSUE_DAY_OF_WEEK"] = (
    df["ISSUE_DATE"].dt.dayofweek
)

df["ISSUE_QUARTER"] = (
    df["ISSUE_DATE"].dt.quarter
)

In [70]:
#remove original date column
df.drop(
    columns=["ISSUE_DATE"],
    inplace=True
)

In [71]:
df.columns.tolist()

['ISN_DOB_BIS_EXTRACT',
 'ECB_VIOLATION_NUMBER',
 'ECB_VIOLATION_STATUS',
 'DOB_VIOLATION_NUMBER',
 'BIN',
 'BORO',
 'BLOCK',
 'LOT',
 'HEARING_DATE',
 'HEARING_TIME',
 'SERVED_DATE',
 'SEVERITY',
 'VIOLATION_TYPE',
 'RESPONDENT_NAME',
 'RESPONDENT_HOUSE_NUMBER',
 'RESPONDENT_STREET',
 'RESPONDENT_CITY',
 'RESPONDENT_ZIP',
 'VIOLATION_DESCRIPTION',
 'PENALITY_IMPOSED',
 'AMOUNT_PAID',
 'BALANCE_DUE',
 'INFRACTION_CODE1',
 'SECTION_LAW_DESCRIPTION1',
 'INFRACTION_CODE2',
 'SECTION_LAW_DESCRIPTION2',
 'INFRACTION_CODE3',
 'SECTION_LAW_DESCRIPTION3',
 'INFRACTION_CODE4',
 'SECTION_LAW_DESCRIPTION4',
 'INFRACTION_CODE5',
 'SECTION_LAW_DESCRIPTION5',
 'INFRACTION_CODE6',
 'SECTION_LAW_DESCRIPTION6',
 'INFRACTION_CODE7',
 'SECTION_LAW_DESCRIPTION7',
 'INFRACTION_CODE8',
 'SECTION_LAW_DESCRIPTION8',
 'INFRACTION_CODE9',
 'SECTION_LAW_DESCRIPTION9',
 'INFRACTION_CODE10',
 'SECTION_LAW_DESCRIPTION10',
 'AGGRAVATED_LEVEL',
 'HEARING_STATUS',
 'CERTIFICATION_STATUS',
 'PREVIOUS_VIOLATION_COUNT',


In [72]:
#remove unnecessary columns
remove_columns = [
    "ECB_VIOLATION_NUMBER",
    "DOB_VIOLATION_NUMBER"
]


df.drop(
    columns=remove_columns,
    errors="ignore",
    inplace=True
)

In [73]:
#Handle Missing Values
categorical_columns = df.select_dtypes(
    include="object"
).columns


for col in categorical_columns:
    df[col] = df[col].fillna(
        "UNKNOWN"
    )

C:\Users\gimha_9pk7du7\AppData\Local\Temp\ipykernel_19428\2484788473.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(


In [74]:
#numeric columns
numerical_columns = df.select_dtypes(
    include=["int64","float64"]
).columns


df[numerical_columns] = (
    df[numerical_columns]
    .fillna(0)
)

In [75]:
#Clean Text Leakage
def clean_description(text):

    text = str(text)

    # remove class 1,2,3 leakage
    text = re.sub(
        r"\bclass\s*[123]\b",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.lower().strip()

In [76]:
df["DESCRIPTION_CLEAN"] = (
    df["VIOLATION_DESCRIPTION"]
    .apply(clean_description)
)

In [77]:
#TF-IDF Encoding
tfidf = TfidfVectorizer(
    max_features=3000,
    stop_words="english"
)


text_features = tfidf.fit_transform(
    df["DESCRIPTION_CLEAN"]
)

In [78]:
#save vectorization
os.makedirs(
    "../models",
    exist_ok=True
)


with open(
    "../models/tfidf.pkl",
    "wb"
) as f:
    pickle.dump(
        tfidf,
        f
    )

In [79]:
#Categorical Encoding
cat_features = [
    "VIOLATION_TYPE",
    "RESPONDENT_CITY",
    "RESPONDENT_ZIP",
    "INFRACTION_CODE1"
]

In [80]:
cat_features = [
    col for col in cat_features
    if col in df.columns
]

In [81]:
#One hot encoder
encoder = OneHotEncoder(
    handle_unknown="ignore"
)


cat_encoded = encoder.fit_transform(
    df[cat_features]
)

In [82]:
#save encoder
with open(
    "../models/categorical_encoder.pkl",
    "wb"
) as f:
    pickle.dump(
        encoder,
        f
    )

In [83]:
#Remove target and text columns
drop_for_X = [
    "SEVERITY",
    "VIOLATION_DESCRIPTION",
    "DESCRIPTION_CLEAN"
]


X_numeric = df.drop(
    columns=drop_for_X,
    errors="ignore"
)

In [84]:
X_numeric = X_numeric.select_dtypes(
    include=["int64","float64"]
)

In [85]:
#Combine Everything
X_final = hstack(
    [
        csr_matrix(
            X_numeric.values
        ),
        cat_encoded,
        text_features
    ]
)

In [87]:
#Encode Target
label_encoder = LabelEncoder()


y = label_encoder.fit_transform(
    df["SEVERITY"]
)

In [88]:
dict(
zip(
label_encoder.classes_,
label_encoder.transform(
label_encoder.classes_
)
)
)

{'CLASS - 1': np.int64(0), 'CLASS - 2': np.int64(1), 'CLASS - 3': np.int64(2)}

In [89]:
os.makedirs(
    "../data/processed",
    exist_ok=True
)

In [90]:
save_npz(
    "../data/processed/X_final.npz",
    X_final
)


np.save(
    "../data/processed/y_final.npy",
    y
)

In [ ]:
# Check duplicate rows
duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

In [ ]:
numerical_columns = df.select_dtypes(
    include=["int64","float64"]
).columns

df[numerical_columns].describe()

In [ ]:
df.columns.tolist()

In [ ]:
df["PENALITY_IMPOSED"].dtype

In [ ]:
df["PENALITY_IMPOSED"].head(10)

In [ ]:
df["PENALITY_IMPOSED"] = (
    df["PENALITY_IMPOSED"]
    .astype(str)
    .str.replace(",", "", regex=False)
)

df["PENALITY_IMPOSED"] = pd.to_numeric(
    df["PENALITY_IMPOSED"],
    errors="coerce"
)

In [ ]:
df["PENALITY_IMPOSED"].dtype

In [ ]:
df[["AMOUNT_PAID", "BALANCE_DUE"]].dtypes

In [ ]:
financial_columns = [
    "AMOUNT_PAID",
    "BALANCE_DUE"
]


for col in financial_columns:
    
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(",", "", regex=False)
    )
    
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

In [ ]:
Q1 = df["PENALITY_IMPOSED"].quantile(0.25)

Q3 = df["PENALITY_IMPOSED"].quantile(0.75)

IQR = Q3 - Q1


lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR


outliers = df[
    (df["PENALITY_IMPOSED"] < lower) |
    (df["PENALITY_IMPOSED"] > upper)
]


print("Number of outliers:", len(outliers))

In [ ]:
df.isnull().sum()